## Libraries

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

from sklearn.ensemble import VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## Feature Generation

In [6]:
def generate_features(train_logs):
    
    chars_per_essay = train_logs[train_logs['activity'] == 'Input'].groupby('id').agg(
        chars_per_essay=('event_id', 'nunique')
    )



    grouped_train_logs = train_logs.groupby('id').agg(
        words_per_essay=('word_count', 'last'),
        max_up_time=('up_time', 'max'),
        min_down_time=('down_time', 'min'),
        events_per_essay=('event_id', 'nunique')
    )

 

    grouped_train_logs['writing_duration'] = grouped_train_logs['max_up_time'] - grouped_train_logs['min_down_time']

    grouped_train_logs = grouped_train_logs.merge(chars_per_essay, left_index=True, right_index=True)

    grouped_train_logs['chars_per_minute'] = 1000 * 60 * grouped_train_logs['chars_per_essay'] / grouped_train_logs['writing_duration']

    ### Pauzy

    train_logs.sort_values(by=['id', 'down_time'], inplace=True)

    train_logs['pause_time'] = (train_logs.groupby('id')['down_time'].shift(0) - train_logs.groupby('id')['up_time'].shift(1)).fillna(0)

  

    train_logs['is_pause'] = train_logs['pause_time'] > 2000

    pause_features = train_logs.groupby('id').agg(
        total_pauses=('is_pause', 'sum'),
        total_pause_time=('pause_time', 'sum')
    )


    grouped_train_logs = grouped_train_logs.merge(pause_features, left_index=True, right_index=True)

  

    grouped_train_logs['perc_pause_time'] = grouped_train_logs['total_pause_time'] / grouped_train_logs['writing_duration']

    grouped_train_logs['pauses_per_words'] = grouped_train_logs['total_pauses'] / grouped_train_logs['words_per_essay']

    ### Aktywności

   

    train_logs['text_change_len'] = train_logs['text_change'].str.len()

  
    train_logs.loc[train_logs['text_change'] == 'NoChange', 'text_change_len'] = 0

  

    activity_per_essay = {}
    activites = ['Remove/Cut', 'Nonproduction', 'Replace', 'Paste']

    for act in activites:
        activity_per_essay[act] = train_logs[train_logs['activity'] == act].groupby('id').agg(
            activity_per_essay=('event_id', 'nunique'),
            activity_len_per_essay=('text_change_len', 'sum')

        )

    activity_per_essay['Remove/Cut']

    for act in activites:
        grouped_train_logs = grouped_train_logs.merge(activity_per_essay[act],
                                                      left_index=True,
                                                      right_index=True,
                                                      suffixes=('', f"_{act}"), how='left')



    grouped_train_logs.rename(columns={'activity_per_essay': 'removes_per_essay',
                                       'activity_len_per_essay': 'removes_len_per_essay'}, inplace=True)

    grouped_train_logs['removes_per_words'] = grouped_train_logs['removes_per_essay'] / grouped_train_logs['words_per_essay']

    grouped_train_logs['replaces_per_words'] = grouped_train_logs['activity_per_essay_Replace'] / grouped_train_logs['words_per_essay']

    grouped_train_logs.fillna(0, inplace=True)

   


    ## Zaawansowane cechy

    grouped_train_logs['chars_per_words'] = grouped_train_logs['chars_per_essay'] / grouped_train_logs['words_per_essay']

    grouped_train_logs.reset_index(inplace=True)



    train_logs = train_logs.merge(grouped_train_logs[['id', 'min_down_time', 'writing_duration', 'words_per_essay']], on='id')

    train_logs['perc_time'] = (train_logs['up_time'] - train_logs['min_down_time']) / train_logs['writing_duration']
    train_logs['perc_words'] = train_logs['word_count'] / train_logs['words_per_essay']


    thres_res = {}

    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    for t in thresholds:
        subset = train_logs[train_logs['perc_words'] >= t]
        subset = pd.DataFrame(subset.groupby('id')['perc_time'].min())
        thres_res[t] = subset

    thres_tab = thres_res[thresholds[0]][['perc_time']]

    for t in thresholds[1:]:
        thres_tab = thres_tab.merge(thres_res[t][['perc_time']],
                                    left_index=True, right_index=True,
                                    suffixes=('', f"_{t}"))

    thres_cols = [f"perc_time_{t}" for t in thresholds]
    thres_tab.columns = thres_cols

 

    grouped_train_logs = grouped_train_logs.merge(thres_tab.reset_index(), on='id')


    ## Burst


    train_logs['group'] = (train_logs['is_pause'].cumsum().shift(fill_value=0))


    grouped = train_logs[~train_logs['is_pause'] & (train_logs['activity'] == 'Input')].groupby(['id', 'group'])

    writing_time_no_pause = grouped['up_time'].last() - grouped['down_time'].first()



    words_no_pause = grouped['word_count'].last() - grouped['word_count'].first()

  

    combined = pd.DataFrame({'writing_time_no_pause': writing_time_no_pause,
                             'words_no_pause': words_no_pause}).reset_index()

    train_logs = train_logs.merge(combined, on=['id', 'group'], how='left')

   

    burst_tab = train_logs.groupby('id')[['words_no_pause', 'writing_time_no_pause']].max().reset_index()

    n_bursts = train_logs.groupby('id')['group'].nunique()

    burst_tab = burst_tab.merge(n_bursts.reset_index())

  

    grouped_train_logs = grouped_train_logs.merge(burst_tab)

   

    grouped_train_logs['bursts_per_words'] = grouped_train_logs['group'] / grouped_train_logs['words_per_essay']
    grouped_train_logs['words_no_pause_per_words'] = grouped_train_logs['words_no_pause'] / grouped_train_logs['words_per_essay']
    grouped_train_logs['writin_time_no_pause_per_time'] = grouped_train_logs['writing_time_no_pause'] / grouped_train_logs['writing_duration']

    return grouped_train_logs

## Dataset Preparation

In [7]:
kaggle_path = Path("/kaggle/input/linking-writing-processes-to-writing-quality")
local_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis")
data_path = kaggle_path if kaggle_path.exists() else local_path

df = pd.read_csv(data_path / "train_logs.csv")
test_df = pd.read_csv(data_path / "test_logs.csv")
df_scores = pd.read_csv(data_path / "train_scores.csv")
sample_submission = pd.read_csv(data_path / "sample_submission.csv")

In [8]:
df_features = generate_features(df)

In [9]:
df_features

,id,words_per_essay,max_up_time,min_down_time,events_per_essay,writing_duration,chars_per_essay,chars_per_minute,total_pauses,total_pause_time,...,perc_time_0.6,perc_time_0.7,perc_time_0.8,perc_time_0.9,words_no_pause,writing_time_no_pause,group,bursts_per_words,words_no_pause_per_words,writin_time_no_pause_per_time
0,001519c8,255,1801969,4526,2557,1797443,2010,67.095313,124,1500200.0,...,0.516710,0.618676,0.651854,0.721284,11.0,23678.0,124,0.486275,0.043137,0.013173
1,0022f953,320,1788969,30623,2454,1758346,1938,66.130329,80,1482955.0,...,0.219058,0.317932,0.383138,0.460980,34.0,47943.0,81,0.253125,0.106250,0.027266
2,0042269b,404,1771669,4441,4136,1767228,3515,119.339440,77,1346027.0,...,0.610536,0.790196,0.820712,0.903792,42.0,67797.0,78,0.193069,0.103960,0.038363
3,0059420b,206,1404469,41395,1556,1363074,1304,57.399672,87,1173478.0,...,0.672673,0.734661,0.830656,0.917422,12.0,30814.0,88,0.427184,0.058252,0.022606
4,0075873a,252,1662472,78470,2531,1584002,1942,73.560513,88,1270300.0,...,0.474403,0.500047,0.539487,0.736932,31.0,32599.0,89,0.353175,0.123016,0.020580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2466,ffb8c745,273,1791649,22467,4739,1769182,3588,121.683354,41,1269512.0,...,0.133427,0.161546,0.175222,0.246015,79.0,103852.0,42,0.153846,0.289377,0.058701
2467,ffbef7e5,438,1799174,21732,2604,1777442,2395,80.846520,90,1563221.0,...,0.452708,0.524862,0.827321,0.919629,35.0,64889.0,91,0.207763,0.079909,0.036507
2468,ffccd6fd,201,1959363,23482,3063,1935881,2849,88.300882,87,1704301.0,...,0.461315,0.517000,0.605986,0.656534,28.0,67968.0,87,0.432836,0.139303,0.035110
2469,ffec5b38,413,1508504,19885,3242,1488619,2895,116.685330,63,1199180.0,...,0.437553,0.522221,0.586659,0.702237,35.0,66817.0,63,0.152542,0.084746,0.044885


In [10]:
df_scores

,id,score
0,001519c8,3.5
1,0022f953,3.5
2,0042269b,6.0
3,0059420b,2.0
4,0075873a,4.0
...,...,...
2466,ffb8c745,3.5
2467,ffbef7e5,4.0
2468,ffccd6fd,1.5
2469,ffec5b38,5.0


In [11]:
X_train_regression = df_features.drop(['id'], axis=1)
y_train_regression = df_scores['score']

## Model Training

In [17]:
scaler = StandardScaler()

X_train_normalized = scaler.fit_transform(X_train_regression)

In [18]:
linear_reg = LinearRegression()
nn_reg = MLPRegressor(max_iter=1000, early_stopping=True, random_state=1)
rf_reg = RandomForestRegressor(max_depth=4, random_state=1)

In [19]:
estimators = [('linear', linear_reg),
              ('neural_network', nn_reg),
              ('rf', rf_reg)]

In [20]:
ereg = VotingRegressor(estimators)

In [21]:
ereg.fit(X_train_normalized, y_train_regression)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('linear', ...), ('neural_network', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100,)"


## Running on the Test Set

In [22]:
df_test_features = generate_features(test_df)

In [23]:
X_test_regression = df_test_features.drop(['id'], axis=1)

In [24]:
X_test_normalized = scaler.transform(X_test_regression)

In [25]:
df_test_features['score'] = ereg.predict(X_test_normalized)

## Publishing Results

In [26]:
submission_path = 'submission.csv'

df_test_features[['id', 'score']].to_csv(submission_path, index=False)